# Set up the Colab QLoRA training environment

In [1]:
!nvidia-smi

Sun Sep 20 06:17:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   65C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install unsloth bitsandbytes wandb

In [3]:
import torch
print(torch.cuda.is_available())

True


In [4]:
from unsloth import FastLanguageModel

model_name = "unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit"  # Phase 0's pick

torch.cuda.reset_peak_memory_stats()
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=1024,
    load_in_4bit=True,
)

mem_load_current = torch.cuda.memory_allocated() / 1e9
mem_load_reserved = torch.cuda.memory_reserved() / 1e9
mem_load_peak = torch.cuda.max_memory_allocated() / 1e9
print(
    f"[After base model load, pre-LoRA] "
    f"allocated={mem_load_current:.2f} GB, "
    f"reserved={mem_load_reserved:.2f} GB, "
    f"peak={mem_load_peak:.2f} GB"
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.7: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

[After base model load, pre-LoRA] allocated=2.10 GB, reserved=2.12 GB, peak=2.12 GB


In [5]:
import wandb
from google.colab import userdata

wandb.login(key=userdata.get("WANDB_API_KEY"))

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: javeedahamed1404 (javeedahamed1404-kumaraguru-college-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [6]:
print("torch.cuda.is_available():", torch.cuda.is_available())
print(f"Pre-LoRA VRAM baseline recorded: {mem_load_current:.2f} GB allocated "
      f"/ {mem_load_reserved:.2f} GB reserved")

torch.cuda.is_available(): True
Pre-LoRA VRAM baseline recorded: 2.10 GB allocated / 2.12 GB reserved


# Configure LoRA adapter hyperparameters

In [7]:
torch.cuda.reset_peak_memory_stats()

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

mem_lora_current = torch.cuda.memory_allocated() / 1e9
mem_lora_reserved = torch.cuda.memory_reserved() / 1e9
mem_lora_peak = torch.cuda.max_memory_allocated() / 1e9
print(
    f"[After LoRA attach] "
    f"allocated={mem_lora_current:.2f} GB, "
    f"reserved={mem_lora_reserved:.2f} GB, "
    f"peak={mem_lora_peak:.2f} GB"
)

# Record trainable parameter count/percentage — should be ~1-2% of total
# params for a 3B model at r=16.
model.print_trainable_parameters()

Unsloth 2026.9.7 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


[After LoRA attach] allocated=2.22 GB, reserved=2.24 GB, peak=2.22 GB
trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


#Build the fast-iteration training subset

In [8]:
from google.colab import drive
drive.mount('/content/drive')

import json

save_dir = "/content/drive/MyDrive/text-to-sql-finetuning/data"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

train_full = load_jsonl(f"{save_dir}/train.jsonl")
print("Full formatted train pool:", len(train_full))

Full formatted train pool: 6440


In [10]:
import random

SUBSET_SEED = 42
SUBSET_SIZE = 1500

random.seed(SUBSET_SEED)
train_subset_raw = random.sample(train_full, SUBSET_SIZE)
print(f"Fast-iteration subset: {len(train_subset_raw)} examples "
      f"(seed={SUBSET_SEED})")

Fast-iteration subset: 1500 examples (seed=42)


In [11]:
from datasets import Dataset

train_subset_ds = Dataset.from_list(
    [{"text": ex["formatted_prompt"]} for ex in train_subset_raw]
)

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=False)

tokenized_lengths_ds = train_subset_ds.map(
    lambda batch: {"n_tokens": [len(ids) for ids in tokenize_fn(batch)["input_ids"]]},
    batched=True,
)

lengths = tokenized_lengths_ds["n_tokens"]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

In [12]:
import statistics

print("Token length stats over the 1500-example subset:")
print(f"  min={min(lengths)}, max={max(lengths)}, "
      f"mean={statistics.mean(lengths):.1f}, median={statistics.median(lengths)}")
print(f"  95th percentile: {sorted(lengths)[int(0.95 * len(lengths))]}")
n_over_1024 = sum(1 for n in lengths if n > 1024)
n_over_512 = sum(1 for n in lengths if n > 512)
print(f"  examples over 512 tokens: {n_over_512} ({n_over_512/len(lengths):.1%})")
print(f"  examples over 1024 tokens: {n_over_1024} ({n_over_1024/len(lengths):.1%})")

Token length stats over the 1500-example subset:
  min=103, max=1853, mean=360.8, median=279.0
  95th percentile: 785
  examples over 512 tokens: 292 (19.5%)
  examples over 1024 tokens: 45 (3.0%)


In [13]:
MAX_SEQ_LENGTH = 1024

# ── Spot-check 3 examples for correct prompt/completion boundaries ─────────
for i in range(3):
    ex = train_subset_raw[i]
    print(f"\n--- Spot check {i} (db_id={ex['db_id']}) ---")
    print(ex["formatted_prompt"])
    print("=" * 60)


--- Spot check 0 (db_id=hospital_1) ---
<|im_start|>system
You are a text-to-SQL model. Given a database schema and a question, output only the SQL query. Do not include explanations or markdown formatting.<|im_end|>
<|im_start|>user
Schema:
CREATE TABLE Physician (EmployeeID number PRIMARY KEY, Name text, Position text, SSN number);
CREATE TABLE Department (DepartmentID number PRIMARY KEY, Name text, Head number);
CREATE TABLE Affiliated_With (Physician number PRIMARY KEY, Department number, PrimaryAffiliation boolean);
CREATE TABLE Procedures (Code number PRIMARY KEY, Name text, Cost number);
CREATE TABLE Trained_In (Physician number PRIMARY KEY, Treatment number, CertificationDate time, CertificationExpires time);
CREATE TABLE Patient (SSN number PRIMARY KEY, Name text, Address text, Phone text, InsuranceID number, PCP number);
CREATE TABLE Nurse (EmployeeID number PRIMARY KEY, Name text, Position text, Registered boolean, SSN number);
CREATE TABLE Appointment (AppointmentID number

# Configure the training run and wire up W&B logging

In [14]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

RUN_NAME = "qwen2.5-coder-3b-lora-r16-a32-fastpass-v1"

training_args = SFTConfig(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,   # effective batch size = 8
    num_train_epochs=1,              # fast pass; Phase 2 scales this up
    learning_rate=2e-4,
    warmup_steps=10,
    logging_steps=1,
    report_to="wandb",
    run_name=RUN_NAME,
    output_dir=f"outputs/{RUN_NAME}",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    packing=False,                   # keep 1 example = 1 sequence, no packing
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_subset_ds,
    args=training_args,
)

# Mask loss to the assistant turn only (see Task 3's note: this is NOT
# automatic in a plain SFTTrainer). Boundary markers match the Qwen2.5
# ChatML-style template seen in the spot-checked examples above.
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/1500 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1500 [00:00<?, ? examples/s]

Unsloth: Removed 45 out of 1500 samples from train_dataset where all labels were -100 (no response marker found, usually truncation). This prevents NaN loss during training.


In [15]:
trainer.args.max_steps = 5
dry_run_result = trainer.train()
print(dry_run_result)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,455 | Num Epochs = 1 | Total steps = 5
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai
`use_return_dict` is deprecated! Use `return_dict` instead!
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:2825: UserWarning: 'has_cuda' is deprecated, please use 'torch.backends.cuda.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:2825: UserWarning: 'has_cudnn' is deprecated, please use 'torch.backends.cudnn.is_available()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:2825: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:2825: UserWarning: 'has_mkldnn' is deprecated, please use '

Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,0.369758
2,0.419681
3,0.522739
4,0.291450
5,0.146066


Unsloth: Restored added_tokens_decoder metadata in outputs/qwen2.5-coder-3b-lora-r16-a32-fastpass-v1/checkpoint-5/tokenizer_config.json.


TrainOutput(global_step=5, training_loss=0.3499386489391327, metrics={'train_runtime': 30.8207, 'train_samples_per_second': 1.298, 'train_steps_per_second': 0.162, 'total_flos': 285575300136960.0, 'train_loss': 0.3499386489391327, 'epoch': 0.027472527472527472})


# Run the first QLoRA fine-tuning pass

In [16]:
import gc

del model, trainer
gc.collect()
torch.cuda.empty_cache()


In [17]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

==((====))==  Unsloth 2026.9.7: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [18]:
training_args = SFTConfig(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,   # ~182 optimizer steps over 1455 examples at
                           # effective batch 8; ~7.9s/step from the dry run
                           # implies ~24 min total — bump to 2 epochs only
                           # if the loss curve hasn't visibly flattened
    learning_rate=2e-4,
    warmup_steps=10,
    logging_steps=1,
    report_to="wandb",
    run_name=RUN_NAME,   # note: same name as the dry run — W&B will start a
                          # fresh run, not resume the dry run's 5 steps, but
                          # check the dashboard for a "-1" suffix if it
                          # auto-deduplicates the name
    output_dir=f"outputs/{RUN_NAME}",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_subset_ds,  # reuse Task 3's dataset object directly
    args=training_args,
)

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/1500 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1500 [00:00<?, ? examples/s]

Unsloth: Removed 45 out of 1500 samples from train_dataset where all labels were -100 (no response marker found, usually truncation). This prevents NaN loss during training.


In [19]:
print(f"Training on {len(trainer.train_dataset)} examples")

Training on 1455 examples


In [20]:
import time
start = time.time()
train_result = trainer.train()
elapsed = time.time() - start

peak_vram = torch.cuda.max_memory_allocated() / 1e9

print(f"Training complete in {elapsed / 60:.1f} min")
print(f"Final training loss: {train_result.training_loss:.4f}")
print(f"Peak VRAM during training: {peak_vram:.2f} GB")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,455 | Num Epochs = 1 | Total steps = 182
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,0.369758
2,0.419681
3,0.523002
4,0.291676
5,0.146086
6,0.170462
7,0.258853
8,0.220655
9,0.257957
10,0.270232


Unsloth: Restored added_tokens_decoder metadata in outputs/qwen2.5-coder-3b-lora-r16-a32-fastpass-v1/checkpoint-182/tokenizer_config.json.


Training complete in 13.4 min
Final training loss: 0.1693
Peak VRAM during training: 4.17 GB


In [21]:
ADAPTER_NAME = "qwen2.5-coder-3b-lora-r16-a32-fastpass-v1"
model.save_pretrained(ADAPTER_NAME)
tokenizer.save_pretrained(ADAPTER_NAME)


Unsloth: Restored added_tokens_decoder metadata in qwen2.5-coder-3b-lora-r16-a32-fastpass-v1/tokenizer_config.json.


('qwen2.5-coder-3b-lora-r16-a32-fastpass-v1/tokenizer_config.json',
 'qwen2.5-coder-3b-lora-r16-a32-fastpass-v1/chat_template.jinja',
 'qwen2.5-coder-3b-lora-r16-a32-fastpass-v1/tokenizer.json')

In [22]:
import shutil

drive_adapter_dir = (
    f"/content/drive/MyDrive/text-to-sql-finetuning/adapters/{ADAPTER_NAME}"
)
shutil.copytree(ADAPTER_NAME, drive_adapter_dir, dirs_exist_ok=True)
print(f"Adapter saved to Drive: {drive_adapter_dir}")

Adapter saved to Drive: /content/drive/MyDrive/text-to-sql-finetuning/adapters/qwen2.5-coder-3b-lora-r16-a32-fastpass-v1


# Funcs from Phase-0

In [28]:
!apt-get install -y git-lfs
!git lfs install

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  git-lfs
0 upgraded, 1 newly installed, 0 to remove and 52 not upgraded.
Need to get 3,908 kB of archives.
After this operation, 11.7 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/universe amd64 git-lfs amd64 3.4.1-1ubuntu0.4 [3,908 kB]
Fetched 3,908 kB in 2s (2,302 kB/s)
Selecting previously unselected package git-lfs.
(Reading database ... 126952 files and directories currently installed.)
Preparing to unpack .../git-lfs_3.4.1-1ubuntu0.4_amd64.deb ...
Unpacking git-lfs (3.4.1-1ubuntu0.4) ...
Setting up git-lfs (3.4.1-1ubuntu0.4) ...
Processing triggers for man-db (2.12.0-4build2) ...
Git LFS initialized.


In [29]:
!git lfs install
!git clone https://huggingface.co/datasets/minktn/spider-data
!ls spider-data

Git LFS initialized.
Cloning into 'spider-data'...
remote: Enumerating objects: 745, done.
remote: Counting objects: 100% (741/741), done.
remote: Compressing objects: 100% (733/733), done.
remote: Total 745 (delta 17), reused 0 (delta 0), pack-reused 4 (from 1)
Receiving objects: 100% (745/745), 4.14 MiB | 4.09 MiB/s, done.
Resolving deltas: 100% (17/17), done.
Filtering content: 100% (450/450), 1.65 GiB | 35.56 MiB/s, done.
README.md  spider_data


In [30]:
from datasets import load_dataset
import json

spider = load_dataset("xlangai/spider")

with open("spider-data/spider_data/tables.json") as f:
    tables = json.load(f)
schema_lookup = {t["db_id"]: t for t in tables}

SPIDER_DB_ROOT = "spider-data/spider_data/database"

# ── Prompt-building functions (unmodified from Phase 0) ─────────────────────
def schema_to_text(db_id, schema_lookup):
    schema = schema_lookup[db_id]
    table_names = schema["table_names_original"]
    column_names = schema["column_names_original"]  # [(table_idx, col_name), ...]
    column_types = schema["column_types"]
    pk_cols = set(schema.get("primary_keys", []))
    fk_pairs = schema.get("foreign_keys", [])

    tables_by_idx = {i: [] for i in range(len(table_names))}
    for col_idx, (tbl_idx, col_name) in enumerate(column_names):
        if tbl_idx == -1:
            continue
        col_type = column_types[col_idx]
        pk_marker = " PRIMARY KEY" if col_idx in pk_cols else ""
        tables_by_idx[tbl_idx].append(f"{col_name} {col_type}{pk_marker}")

    lines = []
    for tbl_idx, tbl_name in enumerate(table_names):
        cols = ", ".join(tables_by_idx[tbl_idx])
        lines.append(f"CREATE TABLE {tbl_name} ({cols});")

    if fk_pairs:
        fk_lines = []
        for from_col_idx, to_col_idx in fk_pairs:
            from_tbl, from_col = column_names[from_col_idx]
            to_tbl, to_col = column_names[to_col_idx]
            fk_lines.append(
                f"-- FK: {table_names[from_tbl]}.{from_col} -> {table_names[to_tbl]}.{to_col}"
            )
        lines.extend(fk_lines)

    return "\n".join(lines)


def build_prompt(schema_text, question, gold_sql=None):
    system_msg = (
        "You are a text-to-SQL model. Given a database schema and a question, "
        "output only the SQL query. Do not include explanations or markdown formatting."
    )
    user_msg = f"Schema:\n{schema_text}\n\nQuestion: {question}"

    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg},
    ]

    if gold_sql is not None:
        messages.append({"role": "assistant", "content": gold_sql})

    return messages


# ── SQL execution + eval harness (unmodified from Phase 0) ─────────────────
import sqlite3
import os


def get_db_connection(db_id):
    db_path = os.path.join(SPIDER_DB_ROOT, db_id, f"{db_id}.sqlite")
    if not os.path.exists(db_path):
        raise FileNotFoundError(f"No sqlite file found for db_id={db_id} at {db_path}")
    uri = f"file:{db_path}?mode=ro"
    conn = sqlite3.connect(uri, uri=True)
    return conn


def run_sql(conn, sql_string):
    try:
        cursor = conn.cursor()
        cursor.execute(sql_string)
        rows = cursor.fetchall()
        return True, rows, None
    except Exception as e:
        return False, [], str(e)


def execution_match(conn, generated_sql, gold_sql):
    gen_success, gen_rows, gen_error = run_sql(conn, generated_sql)
    gold_success, gold_rows, gold_error = run_sql(conn, gold_sql)
    if not gen_success or not gold_success:
        return False
    gen_set = set(tuple(row) for row in gen_rows)
    gold_set = set(tuple(row) for row in gold_rows)
    return gen_set == gold_set


def exact_match(generated_sql, gold_sql):
    def normalize(sql):
        return sql.strip().rstrip(";").strip().lower()
    return normalize(generated_sql) == normalize(gold_sql)


from tqdm.notebook import tqdm


def run_eval_harness(test_examples, generate_fn, results_path="eval_results.json"):
    per_example_results = []
    n_exec_match = 0
    n_exact_match = 0

    for i, ex in enumerate(tqdm(test_examples, desc="Evaluating")):
        db_id = ex["db_id"]
        question = ex["question"]
        gold_sql = ex["gold_sql"]
        schema_text = ex["schema_text"]

        try:
            generated_sql = generate_fn(question, schema_text)
        except Exception as e:
            generated_sql = ""
            print(f"[{i}] generate_fn raised an exception: {e}")

        conn = get_db_connection(db_id)

        exec_ok = execution_match(conn, generated_sql, gold_sql)
        exact_ok = exact_match(generated_sql, gold_sql)

        if exec_ok:
            n_exec_match += 1
        if exact_ok:
            n_exact_match += 1

        per_example_results.append({
            "index": i,
            "db_id": db_id,
            "question": question,
            "generated_sql": generated_sql,
            "gold_sql": gold_sql,
            "execution_match": exec_ok,
            "exact_match": exact_ok,
        })

    n = len(test_examples)
    summary = {
        "execution_accuracy": n_exec_match / n if n > 0 else 0.0,
        "exact_match": n_exact_match / n if n > 0 else 0.0,
        "n": n,
    }

    with open(results_path, "w") as f:
        json.dump({"summary": summary, "per_example": per_example_results}, f, indent=2)

    print("Summary:", summary)
    print(f"Per-example results saved to {results_path}")

    return {"summary": summary, "per_example": per_example_results}


# ── Generation function (unmodified from Phase 0) ───────────────────────────
import re


def extract_sql(raw_output):
    text = raw_output.strip()
    fence_match = re.search(r"```(?:sql)?\s*(.*?)```", text, re.DOTALL)
    if fence_match:
        text = fence_match.group(1).strip()
    sql_start_match = re.search(
        r"(SELECT|WITH|INSERT|UPDATE|DELETE)\b", text, re.IGNORECASE
    )
    if sql_start_match:
        text = text[sql_start_match.start():]
    if ";" in text:
        text = text.split(";")[0] + ";"
    return text.strip()


def generate_sql(question, schema_text, model, tokenizer, max_new_tokens=256):
    messages = build_prompt(schema_text, question)
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    raw_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    return extract_sql(raw_text)

# First trained adapter

In [23]:
import gc

del model, trainer
gc.collect()
torch.cuda.empty_cache()

# ── Fresh base model, for the zero-shot comparison side ────────────────────
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,   # unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(base_model)

==((====))==  Unsloth 2026.9.7: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 2048, padding_idx=151665)
    (layers): ModuleList(
      (0-35): 36 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=True)
          (k_proj): Linear4bit(in_features=2048, out_features=256, bias=True)
          (v_proj): Linear4bit(in_features=2048, out_features=256, bias=True)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=11008, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=11008, bias=False)
          (down_proj): Linear4bit(in_features=11008, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): Qwen

In [24]:
# ── Fresh adapter-loaded model, reloaded from disk (not the trained object) ─
from peft import PeftModel

adapter_model, adapter_tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
adapter_model = PeftModel.from_pretrained(adapter_model, ADAPTER_NAME)
FastLanguageModel.for_inference(adapter_model)

==((====))==  Unsloth 2026.9.7: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 2048, padding_idx=151665)
        (layers): ModuleList(
          (0-35): 36 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

In [31]:
# ── Compare on 3 hand-picked training examples ──────────────────────────────
milestone_examples = train_subset_raw[:3]  # hospital_1, soccer_2, loan_1

for ex in milestone_examples:
    schema_text = ex["schema_text"]
    question = ex["question"]
    gold_sql = ex["gold_sql"]

    zero_shot_sql = generate_sql(question, schema_text, base_model, base_tokenizer)
    adapter_sql = generate_sql(question, schema_text, adapter_model, adapter_tokenizer)

    print("=" * 70)
    print(f"db_id: {ex['db_id']}")
    print(f"Question: {question}")
    print(f"Gold SQL:      {gold_sql}")
    print(f"Zero-shot:     {zero_shot_sql}")
    print(f"Fine-tuned:    {adapter_sql}")
    print(f"Zero-shot matches gold text?  {zero_shot_sql.strip().rstrip(';').lower() == gold_sql.strip().rstrip(';').lower()}")
    print(f"Fine-tuned matches gold text? {adapter_sql.strip().rstrip(';').lower() == gold_sql.strip().rstrip(';').lower()}")

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


db_id: hospital_1
Question: Find the three most expensive procedures.
Gold SQL:      SELECT name FROM procedures ORDER BY cost LIMIT 3
Zero-shot:     SELECT Name FROM Procedures ORDER BY Cost DESC LIMIT 3
Fine-tuned:    SELECT name FROM procedures ORDER BY cost DESC LIMIT 3
Zero-shot matches gold text?  False
Fine-tuned matches gold text? False


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


db_id: soccer_2
Question: How many students, on average, does each college have enrolled?
Gold SQL:      SELECT avg(enr) FROM College
Zero-shot:     SELECT cName, AVG(enr) FROM College GROUP BY cName
Fine-tuned:    SELECT avg(enr) FROM college
Zero-shot matches gold text?  False
Fine-tuned matches gold text? True


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


db_id: loan_1
Question: Find the branch name of the bank that has the most number of customers.
Gold SQL:      SELECT bname FROM bank ORDER BY no_of_customers DESC LIMIT 1
Zero-shot:     SELECT bname FROM bank ORDER BY no_of_customers DESC LIMIT 1
Fine-tuned:    SELECT bname FROM bank ORDER BY no_of_customers DESC LIMIT 1
Zero-shot matches gold text?  True
Fine-tuned matches gold text? True


# Run the eval harness on the fine-tuned checkpoint

In [32]:
def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

test_formatted = load_jsonl(f"{save_dir}/test.jsonl")
print("Loaded test split:", len(test_formatted))

Loaded test split: 1034


In [34]:
random.seed(42)
SUBSET_SIZE = 200

if len(test_formatted) > SUBSET_SIZE:
    test_subset = random.sample(test_formatted, SUBSET_SIZE)
else:
    test_subset = test_formatted

print(f"Evaluating on {len(test_subset)} examples "
      f"(full test set: {len(test_formatted)})")

Evaluating on 200 examples (full test set: 1034)


In [36]:
import os as _os

baseline_path = "baseline_zero_shot_results.json"
if _os.path.exists(baseline_path):
    with open(baseline_path) as f:
        baseline_loaded = json.load(f)
    baseline_questions = {r["question"] for r in baseline_loaded["per_example"]}
    current_questions = {ex["question"] for ex in test_subset}
    print(f"Same example set as Phase 0 baseline? "
          f"{baseline_questions == current_questions}")
else:
    print("Phase 0's baseline_zero_shot_results.json not found in this "
          "runtime — trusting identical seed+order reproduction instead of "
          "diffing example IDs directly. Re-download it from your Phase 0 "
          "session if you want the stronger check.")

Same example set as Phase 0 baseline? True


In [37]:
import warnings

warnings.filterwarnings("ignore", message=".*max_new_tokens.*")


def finetuned_generate_fn(question, schema_text):
    return generate_sql(question, schema_text, adapter_model, adapter_tokenizer)


start = time.time()
finetuned_results = run_eval_harness(
    test_subset,
    finetuned_generate_fn,
    results_path="finetuned_v1_eval_results.json",
)
elapsed = time.time() - start

print(f"Completed {len(test_subset)} examples in {elapsed:.1f}s "
      f"({elapsed / len(test_subset):.2f}s/example)")

print("=== Phase 0 zero-shot baseline ===")
print("Execution accuracy: 61.0%  |  Exact match: 10.5%")
print("=== Phase 1 fine-tuned v1 ===")
print(f"Execution accuracy: {finetuned_results['summary']['execution_accuracy']:.1%}  |  "
      f"Exact match: {finetuned_results['summary']['exact_match']:.1%}")

Evaluating:   0%|          | 0/200 [00:00<?, ?it/s]

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

Summary: {'execution_accuracy': 0.755, 'exact_match': 0.38, 'n': 200}
Per-example results saved to finetuned_v1_eval_results.json
Completed 200 examples in 641.6s (3.21s/example)
=== Phase 0 zero-shot baseline ===
Execution accuracy: 61.0%  |  Exact match: 10.5%
=== Phase 1 fine-tuned v1 ===
Execution accuracy: 75.5%  |  Exact match: 38.0%


# Build the baseline vs. fine-tuned comparison table

In [39]:

import pandas as pd

comparison_rows = [
    {
        "run": "Baseline (Phase 0, zero-shot)",
        "model": "Qwen2.5-Coder-3B-Instruct (4-bit)",
        "lora_rank": "N/A",  # not None — zero-shot has no adapter, this is
                              # a genuine "not applicable," not a missing
                              # value, and stringifying it keeps pandas from
                              # silently upcasting the whole column to float
                              # (which is why 16 was rendering as 16.0)
        "training_examples_seen": 0,
        "execution_accuracy": 0.610,
        "exact_match": 0.105,
    },
    {
        "run": "Fine-tuned v1 (Phase 1)",
        "model": "Qwen2.5-Coder-3B-Instruct (4-bit) + LoRA r=16 a=32",
        "lora_rank": "16",
        "training_examples_seen": 1455,
        "execution_accuracy": finetuned_results["summary"]["execution_accuracy"],
        "exact_match": finetuned_results["summary"]["exact_match"],
    },
]

comparison_df = pd.DataFrame(comparison_rows)

# Explicit deltas, computed on raw floats before formatting as percentages
delta_exec_pts = (
    comparison_df.loc[1, "execution_accuracy"] - comparison_df.loc[0, "execution_accuracy"]
) * 100
delta_exact_pts = (
    comparison_df.loc[1, "exact_match"] - comparison_df.loc[0, "exact_match"]
) * 100

display_df = comparison_df.copy()
display_df["execution_accuracy"] = display_df["execution_accuracy"].map(lambda x: f"{x:.1%}")
display_df["exact_match"] = display_df["exact_match"].map(lambda x: f"{x:.1%}")

print(display_df.to_string(index=False))
print(f"\nDelta (execution accuracy): {delta_exec_pts:+.1f} pts")
print(f"Delta (exact match):        {delta_exact_pts:+.1f} pts")

# Persist — this file is what Phase 2 appends new rows to as its "running
# comparison table" across hyperparameter sweep experiments.
comparison_csv_drive_path = (
    "/content/drive/MyDrive/text-to-sql-finetuning/comparison/"
    "phase0_vs_phase1_v1.csv"
)
os.makedirs(os.path.dirname(comparison_csv_drive_path), exist_ok=True)
comparison_df.to_csv(comparison_csv_drive_path, index=False)
print(f"Comparison table saved to Drive: {comparison_csv_drive_path}")

                          run                                              model lora_rank  training_examples_seen execution_accuracy exact_match
Baseline (Phase 0, zero-shot)                  Qwen2.5-Coder-3B-Instruct (4-bit)       N/A                       0              61.0%       10.5%
      Fine-tuned v1 (Phase 1) Qwen2.5-Coder-3B-Instruct (4-bit) + LoRA r=16 a=32        16                    1455              75.5%       38.0%

Delta (execution accuracy): +14.5 pts
Delta (exact match):        +27.5 pts
Comparison table saved to Drive: /content/drive/MyDrive/text-to-sql-finetuning/comparison/phase0_vs_phase1_v1.csv


# Manual sanity check against known failure patterns

In [42]:
failure_pattern_examples = {
    "Hallucinated/unnecessary joins": {
        "question": "What is the average expected life expectancy for countries in the region of Central Africa?",
        "zero_shot_sql": "SELECT AVG(T2.LifeExpectancy) FROM country AS T1 INNER JOIN countrylanguage AS T2 ON T1.Code = T2.CountryCode WHERE T1.Region = 'Central Africa'",
        "gold_sql": 'SELECT avg(LifeExpectancy) FROM country WHERE Region  =  "Central Africa"',
    },
    "Self-join failures": {
        "question": "Show the names of all of the high schooler Kyle's friends.",
        "zero_shot_sql": "SELECT T2.name FROM Friend AS T1 INNER JOIN Highschooler AS T2 ON T1.student_id = T2.ID WHERE T2.name = 'Kyle'",
        "gold_sql": 'SELECT T3.name FROM Friend AS T1 JOIN Highschooler AS T2 ON T1.student_id  =  T2.id JOIN Highschooler AS T3 ON T1.friend_id  =  T3.id WHERE T2.name  =  "Kyle"',
    },
    "Set operations replaced with JOIN/WHERE": {
        "question": "What are the ids and names of all countries that either have more than 3 car makers or produce fiat model ?",
        "zero_shot_sql": "SELECT T2.CountryId, T1.CountryName FROM countries AS T1 INNER JOIN car_makers AS T2 ON T1.CountryId = T2.CountryId WHERE T2.CountryId IN (SELECT DISTINCT CountryId FROM car_makers WHERE Maker = 'Fiat') OR T1.CountryId IN (SELECT DISTINCT CountryId FROM car_makers GROUP BY CountryId HAVING COUNT(*) > 3)",
        "gold_sql": "select t1.countryid ,  t1.countryname from countries as t1 join car_makers as t2 on t1.countryid  =  t2.country group by t1.countryid having count(*)  >  3 union select t1.countryid ,  t1.countryname from countries as t1 join car_makers as t2 on t1.countryid  =  t2.country join model_list as t3 on t2.id  =  t3.maker where t3.model  =  'fiat';",
    },
    "!= vs NOT IN confusion": {
        "question": "Find the major and age of students who do not have a cat pet.",
        "zero_shot_sql": "SELECT T1.Major, T1.Age FROM Student AS T1 INNER JOIN Has_Pet AS T2 ON T1.StuID = T2.StuID INNER JOIN Pets AS T3 ON T2.PetID = T3.PetID WHERE T3.PetType != 'cat'",
        "gold_sql": "SELECT major ,  age FROM student WHERE stuid NOT IN (SELECT T1.stuid FROM student AS T1 JOIN has_pet AS T2 ON T1.stuid  =  T2.stuid JOIN pets AS T3 ON T3.petid  =  T2.petid WHERE T3.pettype  =  'cat')",
    },
}

with open("finetuned_v1_eval_results.json") as f:
    finetuned_full = json.load(f)

finetuned_by_question = {r["question"]: r for r in finetuned_full["per_example"]}

print("Failure pattern review\n")
for category, info in failure_pattern_examples.items():
    match = finetuned_by_question.get(info["question"])
    print("=" * 70)
    print(f"Category: {category}")
    print(f"Question: {info['question']}")
    print(f"Gold:        {info['gold_sql']}")
    print(f"Zero-shot:   {info['zero_shot_sql']}")
    if match:
        print(f"Fine-tuned:  {match['generated_sql']}")
        print(
            f"Fine-tuned execution_match: {match['execution_match']}  |  "
            f"exact_match: {match['exact_match']}"
        )
    else:
        print(
            "NOT FOUND in this run's 200-example subset — flag this rather "
            "than silently substituting a different example. (Shouldn't "
            "happen given identical seed+sampling, but worth confirming.)"
        )
    print()


Failure pattern review

Category: Hallucinated/unnecessary joins
Question: What is the average expected life expectancy for countries in the region of Central Africa?
Gold:        SELECT avg(LifeExpectancy) FROM country WHERE Region  =  "Central Africa"
Zero-shot:   SELECT AVG(T2.LifeExpectancy) FROM country AS T1 INNER JOIN countrylanguage AS T2 ON T1.Code = T2.CountryCode WHERE T1.Region = 'Central Africa'
Fine-tuned:  SELECT avg(LifeExpectancy) FROM country WHERE Region  =  'Central Africa'
Fine-tuned execution_match: True  |  exact_match: False

Category: Self-join failures
Question: Show the names of all of the high schooler Kyle's friends.
Gold:        SELECT T3.name FROM Friend AS T1 JOIN Highschooler AS T2 ON T1.student_id  =  T2.id JOIN Highschooler AS T3 ON T1.friend_id  =  T3.id WHERE T2.name  =  "Kyle"
Zero-shot:   SELECT T2.name FROM Friend AS T1 INNER JOIN Highschooler AS T2 ON T1.student_id = T2.ID WHERE T2.name = 'Kyle'
Fine-tuned:  SELECT T2.name FROM Friend AS T1 JOIN

# Actual failure-pattern results

**1. Hallucinated/unnecessary joins — FIXED.** Fine-tuned query dropped the
unnecessary countrylanguage join entirely, matching gold's structure.
execution_match=True confirms real correctness; exact_match=False is a pure
quote-style artifact (' vs ") on an otherwise identical query, not a real
error — reinforces Phase 0's own note that exact_match is a weak metric
that penalizes valid textual variation.

**2. Self-join failures — STILL BROKEN, unchanged.** Fine-tuned still only
joins Highschooler once (would return Kyle, not his friends) — identical
structural mistake to zero-shot. Self-joins weren't learned at this
training scale; plausible cause is too few self-join examples in the
1455-example subset to generalize the pattern.

**3. Set-ops as JOIN/WHERE — STILL BROKEN, but differently.** No UNION
produced (core misunderstanding persists), but each OR branch is now a
cleaner, correctly-scoped subquery — a real decomposition improvement. Cost:
a new, irrelevant "JOIN continents" appeared in the outer FROM clause that
wasn't in zero-shot's output at all. Partial improvement + a newly
introduced hallucination in the same query. Worth watching in Phase 2 for
whether "extra unrelated join" becomes a broader pattern at full-dataset
scale (possible early overfitting signal from a small subset).

**4. != vs NOT IN confusion — STILL BROKEN, unchanged.** Same row-level
`!=` filter instead of set-exclusion via NOT IN subquery; only cosmetic
formatting changed (lowercase aliases, quote style).

**Net: 1/4 targeted hard patterns genuinely fixed, 2 unchanged, 1 broken
differently with a new artifact.** An aggregate 75.5%/38.0% alone wouldn't
have surfaced any of this — the model learned the "easy" fix (drop an
unneeded join on a simple single-table aggregate) but the harder structural
patterns (self-joins, UNION, NOT IN subqueries) need more signal than a
1-epoch pass on 1455 examples provided.